# TinyCoder-MCU — Kaggle training notebook

This notebook fine-tunes **Qwen2.5-Coder-1.5B** on STM32N6 embedded-C instruction pairs using QLoRA, then evaluates it.

## Cell 1 — Setup checklist (do this first)

Before running anything, in the notebook's right-hand **Settings** panel:

1. **Accelerator** → select **GPU T4 x2** (we only use one GPU, but this is the free T4 option). A T4 is a Turing card, so we train in **fp16** (it has no bf16 hardware).
2. **Internet** → **On** (needed to `pip install` packages and download the base model from HuggingFace).
3. **Add-ons → Secrets** → add a secret named `ANTHROPIC_API_KEY` *only if* you plan to run dataset building here (normally you build the dataset locally, so you can skip this).
4. **Add data** (right panel) → attach the Kaggle Dataset you created from your local `data/dataset/` folder. Note its mount path — it will be `/kaggle/input/<your-dataset-slug>/`.

**What to look for:** the top-right shows a green GPU indicator once the accelerator is attached.

## Cell 2 — Install the packages Kaggle doesn't ship

`torch`, `transformers`, and `datasets` are pre-installed on Kaggle's GPU image, so we only add the LoRA/quantization stack (reinstalling the base stack risks a CUDA mismatch).

**What to look for:** `Successfully installed … peft bitsandbytes accelerate`. If a later cell throws a bitsandbytes CUDA `ImportError`, do **Restart & clear outputs**, then re-run from here.

In [ ]:
!pip install -q peft bitsandbytes accelerate evaluate rouge-score sacrebleu

## Cell 3 — Clone the project repo

Pulls the training/eval scripts and configs into the Kaggle working directory, then `cd`s into the repo.

**What to look for:** a `Cloning into 'tinycoder-mcu'…` line, and after `%cd` you are inside the repo.

In [ ]:
!git clone https://github.com/Ulysse-Delepoulle/tinycoder-mcu
%cd tinycoder-mcu

## Cell 4 — Train

Runs QLoRA fine-tuning. Update `--data-dir` if your dataset's mount path isn't `/kaggle/input/tinycoder-dataset`.

**What to look for:** `trainable params … trainable%: ~0.5%` (confirms LoRA froze the base), then a decreasing `loss` and periodic `eval_loss`.

In [ ]:
# CUDA_VISIBLE_DEVICES=0 pins training to a SINGLE GPU. Kaggle's "GPU T4 x2"
# exposes two GPUs; HF Trainer would otherwise wrap the model in DataParallel,
# which breaks on a 4-bit bitsandbytes model. We only need one T4 for a 1.5B QLoRA.
!CUDA_VISIBLE_DEVICES=0 python scripts/train.py \
    --data-dir /kaggle/input/tinycoder-dataset \
    --config configs/train_config.yaml \
    --lora-config configs/lora_config.yaml

## Cell 5 — Evaluate

Runs the multi-signal harness against the checkpoint from Cell 4.

**What to look for:** an `EVALUATION SUMMARY` table. The Compilation row will say `skipped` (Kaggle has no ARM cross-compiler — expected); Functional and Similarity still run.

In [ ]:
# CUDA_VISIBLE_DEVICES=0 pins eval to one GPU (stops device_map from splitting the
# model across both T4s, which slows token-by-token generation). Append --limit N
# to evaluate only the first N references for a faster smoke check.
!CUDA_VISIBLE_DEVICES=0 python scripts/evaluate.py \
    --checkpoint /kaggle/working/checkpoints \
    --data-dir /kaggle/input/tinycoder-dataset \
    --output results/eval_results.json

## Cell 6 — Persist the checkpoint

Anything under `/kaggle/working/` is ephemeral until you snapshot it. Click **Save Version → Save & Run All (Commit)** to re-run the notebook and save `/kaggle/working/checkpoints` as the version's Output. From the finished version's **Output** tab you can download the adapter or click **New Dataset** to reuse it in future notebooks.

**What to look for:** the run completes and `checkpoints/` appears under the version's Output.

The saved adapter is only a few MB — LoRA stores adapter weights, not the full base model.